# محرك عثمان · Othman Engine

ارفع تلاوة (صوت أو فيديو)، ويطلع لك السورة والآيات ووقت كل آية وكل كلمة.
Upload a recitation (audio or video) and get the surah, the ayahs and the timing of every ayah and word.

**قبل البدء · Before you start**
1. اقبل شروط النموذج من [صفحته](https://huggingface.co/Muno459/fastconformer-quran)، وأنشئ [token](https://huggingface.co/settings/tokens) (صلاحية Read).
   Accept the model's terms on [its page](https://huggingface.co/Muno459/fastconformer-quran) and create a [token](https://huggingface.co/settings/tokens) (read access).
2. من أيقونة المفتاح 🔑 يسار الصفحة أضف سراً باسم `HF_TOKEN`، وفعّل **Notebook access**.
   In the 🔑 Secrets panel on the left, add `HF_TOKEN` and turn on **Notebook access**.

ثم شغّل الخلايا بالترتيب · Then run the cells in order.

## 1. التثبيت · Install

In [ ]:
import os
from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
if not os.environ["HF_TOKEN"]:
    print("⚠️ HF_TOKEN غير موجود: أضفه من 🔑 وفعّل Notebook access · HF_TOKEN missing: add it in 🔑 with Notebook access")
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!pip install -q "git+{repo}"

## 2. تحميل النموذج ونص المصحف · Download the model and Quran text
مرة واحدة في كل جلسة (نحو دقيقة) · Once per session (about a minute).

In [ ]:
import os

!othman-engine download
if not os.path.exists(os.path.expanduser("~/.othman-engine/models/fastconformer-quran/onnx/model.onnx")):
    raise RuntimeError("لم يكتمل تحميل النموذج؛ اقرأ الرسالة أعلاه · The model was not downloaded; see the message above")


## 3. ارفع التلاوة · Upload the recitation
**الأضمن:** من أيقونة المجلد 📁 يسار الصفحة اضغط زر الرفع ⬆️ واختر الملف، ثم اكتب اسمه في `path` وشغّل الخلية.
إذا تركت `path` فارغاً يظهر زر رفع داخل الخلية (قد لا يعمل في سفاري والآيفون: "Failed to fetch").

**Most reliable:** upload the file from the 📁 Files panel on the left (⬆️ button), type its name in `path`, then run the cell.
Leave `path` empty to get an upload button in the cell instead (it may fail in Safari or on iPhone with "Failed to fetch").

In [ ]:
import os

path = ""  # مثال · example: "recitation.mp3"

if not path:
    from google.colab import files
    path = next(iter(files.upload()))
if not os.path.exists(path):
    raise FileNotFoundError(f"{path} غير موجود · not found. ارفعه من 📁 · upload it from 📁")
print(path)

## 4. التوقيت · Align
لجزء من الملف فقط أضف · For part of the file add: `--start 30 --end 95`

In [ ]:
!othman-engine align "{path}" -o result.json

## 5. النتيجة · Result

In [ ]:
import json

data = json.load(open("result.json", encoding="utf-8"))
for s in data["segments"]:
    ayah = "البسملة" if s["ayah"] == 0 else f"آية {s['ayah']}"
    print(f"{s['surah_name_ar']} {ayah}: {s['start']:.2f} → {s['end']:.2f}  (confidence {s['confidence']})")

print("\nملف النتيجة result.json في 📁 يسار الصفحة (زر النقاط ← Download) · result.json is in the 📁 panel")
try:
    from google.colab import files
    files.download("result.json")
except Exception:
    pass  # التحميل التلقائي قد لا يعمل في بعض المتصفحات؛ الملف في 📁